In [14]:
import requests
import yfinance as yf
import pandas as pd
import numpy as np
import os
import time
import json
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

AV_KEY  = os.environ.get("ALPHAVANTAGE_KEY", "")
TICKERS = ["AAPL", "MSFT", "NVDA", "TSLA", "AMZN", "JPM", "BAC"]

# Alpha Vantage free tier: 25 calls/day, ~5 calls/min
# 2 calls per ticker × 7 tickers = 14 calls total (fits in one day)
DATE_RANGES = [
    ("20220101T0000", "20231231T2359"),   # 2022-2023
    ("20240101T0000", None),              # 2024-now
]

CACHE_DIR = "../../data/raw/av_news_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

if not AV_KEY:
    print("WARNING: ALPHAVANTAGE_KEY not found in .env")
    print("Get a free key at https://www.alphavantage.co/support/#api-key")
else:
    print(f"Alpha Vantage key loaded. Tickers: {TICKERS}")

Alpha Vantage key loaded. Tickers: ['AAPL', 'MSFT', 'NVDA', 'TSLA', 'AMZN', 'JPM', 'BAC']


In [15]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))


def fetch_av_news(symbol, time_from, time_to, api_key, limit=1000):
    """
    One Alpha Vantage NEWS_SENTIMENT call.
    Returns raw article list, or [] on error/rate-limit.
    """
    params = {
        "function": "NEWS_SENTIMENT",
        "tickers":  symbol,
        "time_from": time_from,
        "limit":    limit,
        "sort":     "EARLIEST",
        "apikey":   api_key,
    }
    if time_to:
        params["time_to"] = time_to

    resp = requests.get("https://www.alphavantage.co/query", params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if "feed" not in data:
        msg = data.get("Note") or data.get("Information") or str(data)
        print(f"  AV warning: {msg[:120]}")
        return []
    return data["feed"]


def parse_av_articles(articles, symbol):
    """
    Extract per-day sentiment from AV articles.
    Uses ticker-specific score when available, falls back to overall score.
    AV score range: -1 (bearish) → +1 (bullish)
    """
    rows = []
    for article in articles:
        try:
            date = pd.to_datetime(article["time_published"][:8], format="%Y%m%d").date()
            ticker_sent = next(
                (t for t in article.get("ticker_sentiment", []) if t["ticker"] == symbol),
                None
            )
            score = (
                float(ticker_sent["ticker_sentiment_score"])
                if ticker_sent
                else float(article.get("overall_sentiment_score", 0))
            )
            rows.append({"date": date, "sentiment": score})
        except Exception:
            continue
    return rows

In [16]:
def fetch_all_news(symbol, api_key, date_ranges, sleep_sec=13):
    """
    Fetch all date ranges for one ticker.
    Caches raw JSON to disk so re-runs skip already-fetched chunks.
    sleep_sec=13 → ~4.6 calls/min, safely under AV's 5 calls/min limit.
    """
    all_rows = []
    for time_from, time_to in date_ranges:
        label    = f"{time_from[:8]}-{time_to[:8] if time_to else 'now'}"
        cache_path = f"{CACHE_DIR}/{symbol}_{label}.json"

        if os.path.exists(cache_path):
            with open(cache_path) as f:
                articles = json.load(f)
            print(f"  [cache] {label}: {len(articles)} articles")
        else:
            articles = fetch_av_news(symbol, time_from, time_to, api_key)
            with open(cache_path, "w") as f:
                json.dump(articles, f)
            print(f"  [fetch] {label}: {len(articles)} articles")
            time.sleep(sleep_sec)

        all_rows.extend(parse_av_articles(articles, symbol))

    print(f"  Total scored rows: {len(all_rows)}")
    return all_rows

In [17]:
def build_features(symbol, api_key, date_ranges):
    # 1. Price + technical features
    df = yf.download(symbol, start="2022-01-01", progress=False)
    df.columns = df.columns.get_level_values(0)
    df["momentum"]   = df["Close"].pct_change(20)
    df["volatility"] = df["Close"].rolling(20).std()
    df["rsi"]        = compute_rsi(df["Close"])
    df["ma_diff"]    = df["Close"] / df["Close"].rolling(50).mean() - 1

    # 2. Fetch & aggregate sentiment
    if api_key:
        news_rows = fetch_all_news(symbol, api_key, date_ranges)
    else:
        print("  No ALPHAVANTAGE_KEY — sentiment will be 0")
        news_rows = []

    if news_rows:
        df_sent    = pd.DataFrame(news_rows)
        daily_sent = df_sent.groupby("date")["sentiment"].agg(
            sentiment_mean="mean",
            sentiment_count="count",
            sentiment_std="std",
        ).fillna(0)
    else:
        daily_sent = pd.DataFrame(
            columns=["sentiment_mean", "sentiment_count", "sentiment_std"]
        )

    # 3. Merge
    df.index = pd.to_datetime(df.index).date
    df = df.merge(daily_sent, left_index=True, right_index=True, how="left").fillna(0)

    # 4. Lagged sentiment features
    df["sentiment_change"] = df["sentiment_mean"].diff()
    df["sentiment_3d"]     = df["sentiment_mean"].rolling(3).mean()
    df["sentiment_7d"]     = df["sentiment_mean"].rolling(7).mean()

    # 5. Target
    df["target"] = (df["Close"].shift(-10) / df["Close"] - 1 > 0.02).astype(int)

    return df

In [18]:
os.makedirs("../../data/processed/sentiment", exist_ok=True)

for symbol in TICKERS:
    print(f"\n{'='*40}")
    print(f"Processing: {symbol}")
    df = build_features(symbol, AV_KEY, DATE_RANGES)
    out_path = f"../../data/processed/sentiment/sentiment_features_{symbol}.csv"
    df.to_csv(out_path)
    print(f"Saved → {out_path}  shape={df.shape}")

print("\nAll tickers done!")


Processing: AAPL
  [cache] 20220101-20231231: 1000 articles
  [cache] 20240101-now: 1000 articles
  Total scored rows: 2000
Saved → ../../data/processed/sentiment/sentiment_features_AAPL.csv  shape=(1088, 16)

Processing: MSFT
  [cache] 20220101-20231231: 1000 articles
  [cache] 20240101-now: 1000 articles
  Total scored rows: 2000
Saved → ../../data/processed/sentiment/sentiment_features_MSFT.csv  shape=(1088, 16)

Processing: NVDA
  [cache] 20220101-20231231: 1000 articles
  [cache] 20240101-now: 1000 articles
  Total scored rows: 2000
Saved → ../../data/processed/sentiment/sentiment_features_NVDA.csv  shape=(1088, 16)

Processing: TSLA
  [cache] 20220101-20231231: 1000 articles
  [cache] 20240101-now: 1000 articles
  Total scored rows: 2000
Saved → ../../data/processed/sentiment/sentiment_features_TSLA.csv  shape=(1088, 16)

Processing: AMZN
  [cache] 20220101-20231231: 1000 articles
  [cache] 20240101-now: 1000 articles
  Total scored rows: 2000
Saved → ../../data/processed/sentim